In [ ]:
from Bio import SeqIO
import numpy as np
import pandas as pd
from collections import Counter

In [50]:
proteins = []
file_path = 'deeploc_data.fasta'
file_type = 'fasta'
for protein in SeqIO.parse(file_path, file_type):
    protein_seq = str(protein.seq)

    # header
    header = protein.description

    proteins.append({
        "protein_id": protein.id,
        "sequence": protein_seq,
        "header": header
    })

df = pd.DataFrame(proteins)
print(df.head())

  protein_id                                           sequence  \
0     Q9H400  MGLPVSWAPPALWVLGCCALLLSLWALCTACRRPEDAVAPRKRARR...   
1     Q5I0E9  MEVLEEPAPGPGGADAAERRGLRRLLLSGFQEELRALLVLAGPAFL...   
2     P63033  MMKTLSSGNCTLNVPAKNSYRMVVLGASRVGKSSIVSRFLNGRFED...   
3     Q9NR71  MAKRTFSNLETFLIFLLVMMSAITVALLSLLFITSGTIENHKDLGG...   
4     Q86XT9  MGNCQAGHNLHLCLAHHPPLVCATLILLLLGLSGLGLGSFLLTHRT...   

                        header  
0  Q9H400 Cell.membrane-M test  
1       Q5I0E9 Cell.membrane-M  
2       P63033 Cell.membrane-M  
3       Q9NR71 Cell.membrane-M  
4       Q86XT9 Cell.membrane-M  


In [51]:
def parse_header(header):
    '''
    Extract the subcellular location (output label), membrane_status, and train/test label from the header
    Args:
        header(str): header for row
    Returns:
        Tuple of subcellular loc, membrane status, and train test label

    '''
    header_parts = header.split()

    subcellular_location = header_parts[1][:-2]
    membrane_status = header_parts[1][-1]

    if len(header_parts) > 2:
        train_test = header_parts[2]
    else:
        train_test = 'train'
    
    return (subcellular_location, membrane_status, train_test)

new_cols = df['header'].apply(parse_header).apply(pd.Series)
df[['subcellular_location', 'membrane_status', 'train_test']] = new_cols

# drop headers column: info has been parsed into new cols
df.drop(columns=['header'], inplace=True)

print(df.head())

  protein_id                                           sequence  \
0     Q9H400  MGLPVSWAPPALWVLGCCALLLSLWALCTACRRPEDAVAPRKRARR...   
1     Q5I0E9  MEVLEEPAPGPGGADAAERRGLRRLLLSGFQEELRALLVLAGPAFL...   
2     P63033  MMKTLSSGNCTLNVPAKNSYRMVVLGASRVGKSSIVSRFLNGRFED...   
3     Q9NR71  MAKRTFSNLETFLIFLLVMMSAITVALLSLLFITSGTIENHKDLGG...   
4     Q86XT9  MGNCQAGHNLHLCLAHHPPLVCATLILLLLGLSGLGLGSFLLTHRT...   

  subcellular_location membrane_status train_test  
0        Cell.membrane               M       test  
1        Cell.membrane               M      train  
2        Cell.membrane               M      train  
3        Cell.membrane               M      train  
4        Cell.membrane               M      train  


In [52]:
df['train_test'].value_counts()

train_test
train    11231
test      2773
Name: count, dtype: int64

In [53]:
# check missing values
print(df.isnull().sum())
# check duplicates
print("Duplicate IDs:", df['protein_id'].duplicated().sum())
print("Duplicate sequences:", df['sequence'].duplicated().sum())

protein_id              0
sequence                0
subcellular_location    0
membrane_status         0
train_test              0
dtype: int64
Duplicate IDs: 0
Duplicate sequences: 55


In [62]:
# check unusual amino acids
usual_amino_acids = set("ACDEFGHIKLMNPQRSTVWY"),
unusual_amino_acids = set("XBZUOJ")

# check if a sequence contains unusual amino acids
def count_unusual(seq):
    count = 0
    for aa in seq:
        if aa in unusual_amino_acids:
            count += 1
    return count

df['num_unusual_aa'] = df['sequence'].apply(count_unusual)
print((df['num_unusual_aa'] > 0).sum())

34


In [63]:
from collections import Counter

unusual_rows = df[df['num_unusual_aa'] > 0]

counts = Counter()

for seq in unusual_rows['sequence']:
    for aa in seq:
        if aa in unusual_amino_acids:
            counts[aa] += 1

print(counts)

Counter({'X': 76, 'U': 30, 'Z': 4, 'B': 2})


In [54]:
# check variation in sequence lengths
df['seq_length'] = df['sequence'].str.len()
print(df['seq_length'].describe())

count    14004.00000
mean       522.44166
std        448.06671
min         40.00000
25%        249.00000
50%        421.00000
75%        649.00000
max      13100.00000
Name: seq_length, dtype: float64


In [55]:
# inspect duplicates
duplicates = df[df.duplicated('sequence', keep=False)]
duplicates = duplicates.sort_values('sequence')
print(f"duplicate labels: {duplicates['train_test'].value_counts()}")

print(duplicates[
    ['protein_id', 'subcellular_location',
     'membrane_status', 'train_test']
])

cols = ['subcellular_location', 'membrane_status', 'train_test']

# check which duplicates have mismatch data: potential problem
conflicts = df.groupby('sequence')[cols].nunique()
conflicting_sequences = conflicts[(conflicts > 1).any(axis=1)]

# retreive df rows of duplicates
conflicting_rows = df[df['sequence'].isin(conflicting_sequences.index)]

print(conflicting_rows[['protein_id', 'subcellular_location', 'membrane_status', 'train_test']])

duplicate labels: train_test
train    95
test     10
Name: count, dtype: int64
      protein_id subcellular_location membrane_status train_test
11674     P68164              Plastid               U      train
11675     P68163              Plastid               U      train
13583     P0C892        Extracellular               S      train
13018     Q86QV0        Extracellular               S      train
13898     P83288        Extracellular               S      train
...          ...                  ...             ...        ...
1628      P0CX27            Cytoplasm               S      train
13421     Q25619        Extracellular               S      train
13962     Q8WT57        Extracellular               S      train
13008     P85174        Extracellular               S      train
13654     P85436        Extracellular               S      train

[105 rows x 4 columns]
Empty DataFrame
Columns: [protein_id, subcellular_location, membrane_status, train_test]
Index: []


In [56]:
# separate train and test dataframes
train_df = df[df['train_test'] == 'train'].copy()
test_df = df[df['train_test'] == 'test'].copy()

# remove duplicates from training data
train_df = train_df.drop_duplicates(subset='sequence')

# train and test dataset sizes
print("Training set size:", len(train_df))
print("Test set size:", len(test_df))
train_percent = 100 * len(train_df) / len(df)
test_percent = 100 * len(test_df) / len(df)
print("Train/Test Split:", train_percent, ':', test_percent)

Training set size: 11181
Test set size: 2773
Train/Test Split: 79.84147386461011 : 19.801485289917167


In [58]:
train_df.head()

,protein_id,sequence,subcellular_location,membrane_status,train_test,seq_length
1,Q5I0E9,MEVLEEPAPGPGGADAAERRGLRRLLLSGFQEELRALLVLAGPAFL...,Cell.membrane,M,train,566
2,P63033,MMKTLSSGNCTLNVPAKNSYRMVVLGASRVGKSSIVSRFLNGRFED...,Cell.membrane,M,train,266
3,Q9NR71,MAKRTFSNLETFLIFLLVMMSAITVALLSLLFITSGTIENHKDLGG...,Cell.membrane,M,train,780
4,Q86XT9,MGNCQAGHNLHLCLAHHPPLVCATLILLLLGLSGLGLGSFLLTHRT...,Cell.membrane,M,train,240
5,A2CI98,MDPSKQGTLNRVENSVYRTAFKLRSVQTLCQLDLMDSFLIQQVLWR...,Cell.membrane,M,train,653


In [59]:
test_df.head()

,protein_id,sequence,subcellular_location,membrane_status,train_test,seq_length
0,Q9H400,MGLPVSWAPPALWVLGCCALLLSLWALCTACRRPEDAVAPRKRARR...,Cell.membrane,M,test,295
9,P83456,AGFPEQEPEPKFWNDWAQKTLDKALSLQTLNKNKAQNLILFLGDGM...,Cell.membrane,M,test,477
15,Q9GL77,MEDEAVLDRGASFLKHVCDEEEVEGHHTIYIGVHVPKSYRRRRRHK...,Cell.membrane,M,test,1079
20,Q03445,MHSRLKFLAYLHFICASSIFWPEFSSAQQQQQTVSLTEKIPLGAIF...,Cell.membrane,M,test,991
23,P35525,MAAATAAAATVAGEGMEPRALQYEQTLMYGRYTQELGAFAKEEAAR...,Cell.membrane,M,test,907
